# Ch 12. 검색 품질 개선

이 노트북은 [AI Assistant Engineering - Part 3, Ch 12](https://desty.github.io/study-ai-assistant-engineering/part3/12-retrieval-quality/) 의 실습용 보조 자료입니다.

## 주제
- **답변 품질의 70%는 검색 품질** — 생성으로는 못 메꾸는 이유
- 검색 실패의 **5가지 유형** (recall · precision · 랭킹 · metadata · chunk 경계)
- **BM25 + Dense** 하이브리드 검색 (RRF 병합)
- **Cross-encoder reranker** — 순위 재배열로 정밀도 올리기
- **MMR** (다양성) 과 **metadata filter**

In [ ]:
!pip install -q chromadb openai rank-bm25

In [ ]:
import os
from getpass import getpass

for k in ['OPENAI_API_KEY']:
    if not os.getenv(k):
        os.environ[k] = getpass(f'{k}: ')

## 1. 개념 — 검색이 답변의 상한을 정한다

Ch 11 의 Query 파이프라인에서 **검색이 무관한 문서**를 뽑으면 LLM은 **거기서 답을 만들려고 시도**합니다.

> **생성은 검색이 놓친 걸 복구하지 못한다.**

이 챕터는 **검색 품질 개선**의 도구상자.

## 2. 검색 실패의 5가지 유형

| 유형 | 증상 | 원인 |
|---|---|---|
| **Recall 부족** | 관련 문서가 top-k에 아예 없음 | 쿼리-문서 표현 차이, chunk 경계 문제 |
| **Precision 부족** | 무관 문서가 top-k에 많이 섞임 | ANN 점수만으로 정렬, 다양성 없음 |
| **랭킹 오류** | 관련 문서가 있지만 낮은 순위 | Dense 만의 한계 |
| **Metadata 무시** | 오래된·비공개 문서가 top-k | updated_at · owner 필터 안 걸림 |
| **Chunk 경계** | 답이 두 chunk에 걸쳐 있음 | 고정 길이 자르기 |

각 유형마다 **다른 도구**가 필요.

## 4. 최소 예제 — Dense vs BM25 vs Hybrid 비교

In [ ]:
from openai import OpenAI
from rank_bm25 import BM25Okapi
import numpy as np

openai = OpenAI()

docs = [
    "환불은 구매 후 7일 이내, 팀장 승인 필요.",
    "배송 정책: 영업일 2~3일, 도서산간 +2일.",
    "포인트는 구매 금액의 1% 적립, 3개월 이내 사용.",
    "AS 기간은 제품 구매일로부터 1년.",
    "환불 불가 품목: 맞춤 제작 상품, 개봉된 소프트웨어.",
]

def embed(texts):
    r = openai.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([d.embedding for d in r.data])

doc_vecs = embed(docs)

tokenized = [d.split() for d in docs]
bm25 = BM25Okapi(tokenized)

query = "돈을 돌려받고 싶은데요"
q_vec = embed([query])[0]

dense_scores = doc_vecs @ q_vec
dense_top = np.argsort(-dense_scores)[:3]

bm25_scores = bm25.get_scores(query.split())
bm25_top = np.argsort(-bm25_scores)[:3]

print("=== Dense Top-3 ===")
for i in dense_top:
    print(f"{dense_scores[i]:.3f}  {docs[i]}")

print("\n=== BM25 Top-3 ===")
for i in bm25_top:
    print(f"{bm25_scores[i]:.3f}  {docs[i]}")

## 5. 실전 튜토리얼 - Hybrid 검색

In [ ]:
def rrf_merge(ranked_lists: list[list[int]], k: int = 60) -> dict:
    """각 리스트의 순위를 1/(k+rank) 로 환산해 합산."""
    scores = {}
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank + 1)
    return scores

dense_top20 = list(np.argsort(-dense_scores)[:20])
bm25_top20  = list(np.argsort(-bm25_scores)[:20])
merged = rrf_merge([dense_top20, bm25_top20])
final_top5 = sorted(merged, key=merged.get, reverse=True)[:5]

print("=== Hybrid (RRF) Top-5 ===")
for doc_id in final_top5:
    print(f"{merged[doc_id]:.4f}  {docs[doc_id]}")

## 6. 자주 깨지는 포인트

### 실수 1. Chunk 를 무조건 작게
recall 은 올라가지만 맥락이 잘려 답변 품질은 오히려 하락 가능.

**대응**: chunk_size=512, overlap=64에서 시작. 실패 분석에 따라 조정.

### 실수 2. Rerank 를 top-500 에 적용
Cross-encoder 는 느림. top-500 rerank 하면 p95 지연이 5~10초.

**대응**: Dense/BM25 로 top-20~50 먼저 · 그 중에서만 rerank.

### 실수 3. Metadata 필터 없이 최신 문서 기대
**대응**: updated_at 필터 + 최신성 boosting.

### 실수 4. BM25 한국어 토큰화
**대응**: 형태소 분석기 (kiwi, KoNLPy) 로 토큰화.

### 실수 5. Rerank 비용·지연 미측정
**대응**: 평균 호출당 비용·지연 측정.

## 8. 확인 문제

1. dense_vs_bm25 를 한국어 문장 10건, 영어 10건으로 돌려 유형별 강점 정리
2. RRF 병합 결과와 각 검색기 단독의 top-5 를 비교
3. chunk_size 256, 512, 1024 로 변경하며 precision/recall 변화 기록
4. 오래된 문서를 top-1 으로 올라오도록 만든 뒤, updated_at filter 로 해결
5. 다양성(MMR) 적용 전후 top-5의 정보 밀도 비교

**다음 챕터** → [Ch 13. Advanced RAG](https://desty.github.io/study-ai-assistant-engineering/part3/13-advanced-rag/)

기본기는 완성. 이제 HyDE · Self-RAG · GraphRAG · Agentic RAG 등 논문급 기법들.